# 01 — Spark on ObjectScale
**Objective:** Spark 4.2 reads and writes ObjectScale S3 via the S3A connector.

**Customer sees:** a partitioned dataset written to OBS, listed in the OBS UI, read back and queried.
**TC:** TC1.1–TC1.6

## Step 1 — Session
Build the SparkSession and print versions (masked credentials).

In [ ]:
import time
import matplotlib.pyplot as plt
import pandas as pd
from poc import config, demo_data, results, throughput, viz
import poc.spark_helpers as sh

spark = sh.build_session("obs-poc")
viz.table(pd.DataFrame(sh.versions(spark).items(), columns=["component", "version"]))
print("run id:", results.run_id(), "| session identities:", sh.identities(spark))

**Presenter notes** (hidden)

Same session configuration as the smoke test — one source of truth in poc/spark_helpers.py.

## Step 2 — Generate the dataset (TC1.1)
Synthetic sales orders generated **on the Spark side** from the row id — deterministic, 30 daily partitions from a fixed base date. Row count scales with `DEMO_SCALE_GB`.

In [ ]:
t0 = time.time()
df = demo_data.orders_df(spark)
rows = df.count()
want = demo_data.n_rows_for_scale()
res = results.write("TC1.1", "dataset generated", rows == want,
                    {"rows": rows, "expected_rows": want,
                     "demo_scale_gb": config.DEMO_SCALE_GB,
                     "base_date": config.DEMO_BASE_DATE,
                     "seconds": round(time.time() - t0, 1)})
df.show(5)
viz.banner(res["status"], "TC1.1",
           f"{rows:,} rows (DEMO_SCALE_GB={config.DEMO_SCALE_GB:g}), "
           f"first partition {config.DEMO_BASE_DATE}")

**Presenter notes** (hidden)

Every value is a hash of the row id — the same data on every run, nothing materialised on the driver. DEMO_SCALE_GB counts uncompressed row data; see reports/environment.md for the sizing.

## Step 3 — Write Parquet partitioned by date (TC1.2)
Data lands in `poc-spark-data/<run_id>/raw/orders` — one partition per order_date.

In [ ]:
path = demo_data.data_path()
t0 = time.time()
df.write.mode("overwrite").partitionBy("order_date").parquet(path)
secs = round(time.time() - t0, 1)
marker, _ = demo_data.list_objects(config.BUCKET_DATA,
                                   f"{results.run_id()}/raw/orders/_SUCCESS")
res = results.write("TC1.2", "write partitioned parquet", len(marker) == 1,
                    {"path": path, "rows": rows, "seconds": secs,
                     "success_marker": len(marker) == 1})
viz.banner(res["status"], "TC1.2", f"{res['metrics']['seconds']}s -> {path}")

**Presenter notes** (hidden)

Show the Spark UI stages during the write. Parquet uses Spark's ParquetOutputCommitter (rename-based) on this cluster; the S3A magic committer is active for the other formats (README, 'Spark configuration').

## Step 4 — Objects on ObjectScale (TC1.3)
List the objects Spark just wrote, then **pause and show the bucket in the ObjectScale UI** — same files, visible in the storage console.

In [ ]:
objs, total = demo_data.list_objects(config.BUCKET_DATA, f"{results.run_id()}/raw/")
viz.object_table([(o["Key"], o["Size"]) for o in objs])
res = results.write("TC1.3", "objects listed in bucket", total["objects"] > 0,
                    {**total, "mib": round(total["bytes"] / 2**20, 1)})
viz.banner(res["status"], "TC1.3",
           f"{total['objects']} objects, {total['bytes'] / 2**20:.1f} MiB "
           f"— show the OBS UI now")

**Presenter notes** (hidden)

UI: bucket poc-spark-data, prefix = the run id printed in Step 1.

## Step 5 — Read back + business queries (TC1.4)
Read the Parquet dataset back and answer three business questions: revenue by day, top customers, orders by status.

In [ ]:
spark.read.parquet(path).createOrReplaceTempView("orders")
q1 = spark.sql("SELECT order_date, round(sum(amount*qty),2) rev FROM orders GROUP BY 1 ORDER BY 1").toPandas()
q2 = spark.sql("SELECT customer, round(sum(amount*qty),2) rev FROM orders GROUP BY 1 ORDER BY rev DESC LIMIT 10").toPandas()
q3 = spark.sql("SELECT status, count(*) n FROM orders GROUP BY 1 ORDER BY 1").toPandas()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
q1.plot.bar(x="order_date", y="rev", ax=ax[0], legend=False); ax[0].set_title("revenue/day")
q2.plot.bar(x="customer", y="rev", ax=ax[1], legend=False); ax[1].set_title("top customers")
res = results.write("TC1.4", "read+queries", len(q1) > 0 and len(q2) > 0,
                    {"days": len(q1), "top": q2.iloc[0].customer})
viz.table(q3); plt.show(); viz.banner(res["status"], "TC1.4")

**Presenter notes** (hidden)

Queries hit OBS via S3A on every executor.

## Step 6 — Integrity check (TC1.5)
Row count and column checksums of the stored dataset must equal the source — no data lost or altered.

In [ ]:
agg = ["count(*) AS n", "sum(qty) AS qty", "round(sum(amount), 2) AS amount"]
src = df.selectExpr(*agg).first().asDict()
dst = spark.table("orders").selectExpr(*agg).first().asDict()
res = results.write("TC1.5", "integrity", src == dst, {"src": src, "dst": dst})
viz.table(pd.DataFrame([src, dst], index=["source", "stored"]))
viz.banner(res["status"], "TC1.5", f"{src['n']:,} rows, checksums "
           f"{'match' if src == dst else 'DIFFER'}")

**Presenter notes** (hidden)

Count plus sum(qty) and sum(amount) — a stronger check than a count.

## Step 7 — Multipart throughput vs concurrency (TC1.6)
Same 256 MiB object uploaded at 1, 2, 4, 8 and 16 parallel part streams — shows how concurrency scales sustained write throughput.

In [ ]:
levels = throughput.multipart_sweep(
    config.BUCKET_THROTTLE, f"{results.run_id()}/multipart",
    levels=(1, 2, 4, 8, 16), total_mb=256, part_mb=16)
viz.table(pd.DataFrame(levels))
res = results.write("TC1.6", "multipart write concurrency sweep",
                    all(r["mbs"] > 0 for r in levels),
                    {"levels": levels, "peak_mbs": max(r["mbs"] for r in levels)})
viz.banner(res["status"], "TC1.6", f"peak {res['metrics']['peak_mbs']:.0f} MB/s")

**Presenter notes** (hidden)

Objects land under poc-throttle/<run_id>/multipart/ — cleaned by 99_reset. The client's connection pool is sized for 16 streams.

## Step 8 — Throughput chart
Throughput vs concurrency — the ingest-scaling story.

In [ ]:
sw = pd.DataFrame(res["metrics"]["levels"]).set_index("concurrency")
sw.plot.line(y="mbs", marker="o", figsize=(6, 3.5), legend=False)
plt.xlabel("concurrent part streams"); plt.ylabel("MB/s")
plt.title("S3 multipart write throughput vs concurrency"); plt.grid(alpha=.3)
plt.show()
spark.stop()

**Presenter notes** (hidden)

Scaling tapers when a host/NIC limit is hit — expected on a 2-core lab VM.